# Step 6 — Tuning, Ensemble and Significance Testing

**Starting point (Step 5).** The neural ranker ties LightGBM on NDCG@10 (0.4154 vs 0.4159). Each model wins in different places: the neural model on cold slates and purchases, LightGBM on warm slates. The neural model also stopped improving only because training ended at epoch 4.

| Part | What it does | Selected on |
|---|---|---|
| 1. Neural tuning | Up to 12 epochs with early stopping and a cosine learning-rate schedule; a small grid over history length, width and listwise-loss weight; the best setting retrained with 3 seeds and averaged | validation NDCG@10 |
| 2. LightGBM tuning | Higher credit for purchases in the ranking objective (`label_gain`), then two hyperparameter variants | validation NDCG@10 (Purchase-HR@10 reported) |
| 3. Ensemble | Within each slate, standardise both models' scores and take a weighted average; weight and method chosen on validation | validation NDCG@10 |
| 4. Significance | Paired bootstrap over test slates for every comparison that matters, plus mean ± sd across neural seeds | — |
| 5. Business translation | Session-conversion estimates from the calibrated purchase probabilities; where the purchased item lands under each ranking | — |

**The test set is used only after every choice is fixed on validation**, so the final numbers are honest.

**Runtime:** Colab **L4 GPU**, about 1–1.5 h (LightGBM runs on the CPU cores of the same runtime). The LSA embedding cache from Step 5 is reused.

In [ ]:
import os, sys, time, gc, copy
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb, warnings
import torch, torch.nn as nn, torch.nn.functional as Fn
if sys.platform == 'darwin':
    torch.set_num_threads(1)     # macOS only: torch and LightGBM ship separate OpenMP runtimes that segfault together
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv, pyarrow.parquet as pq, pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import svds
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'
SEED, K, L_MAX = 42, 64, 100
if SMOKE:
    MAX_EPOCHS, PATIENCE, BATCH, SEEDS, LGB_TREES = 2, 1, 32, [42, 43], 40
else:
    MAX_EPOCHS, PATIENCE, BATCH, SEEDS, LGB_TREES = 12, 2, 48, [42, 43, 44], 600
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
AMP_DEV = 'cuda' if USE_AMP else 'cpu'
print(f'device {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if USE_AMP else '') + (' | SMOKE TEST' if SMOKE else ''))
if DEVICE == 'cpu' and not SMOKE:
    print('WARNING: no GPU. Runtime > Change runtime type > L4 GPU.')

FEATURES_LEAN = [
    'slate_pct_pop', 'i_users_click_7d', 'uc_days_since_any', 'i_users_cart_90d', 'i_users_ord_365d',
    'q_name_tfidf_sim', 'slate_len', 'ubrand_days_since_any', 'c3_click_7d', 'slate_cat3_share',
    'u_days_since_click', 'qc_cat3_prior_max', 'uc_click_7d', 'ucat3_days_since_any', 'q_name_match_frac',
    'slate_pct_text', 'slate_pct_sem', 'c3_ord_per_cart_90d', 'q_meta_tfidf_sim', 'q_sem_lsa_sim',
    'u_days_since_cart', 'q_n_terms', 'uc_cart_90d', 'ucat2_click_7d', 'uc_ord_365d',
    'u_ord_365d', 'u_click_7d', 'u_days_since_ord', 'u_cart_per_click_7d', 'u_ord_per_cart_90d',
    'u_queryless_share_7d', 'u_query_repeat_rate']
SEASONAL = ['cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share']
FEATURES_ALL = [c for c in pq.read_schema(OUT_DIR / 'features.parquet').names
                if c not in ('user', 'wid', 'label', 'split') and c not in SEASONAL]
missing = [f for f in FEATURES_LEAN if f not in FEATURES_ALL]
if missing:
    print(f'WARNING: {missing} not in features.parquet (run Steps 4 and 4b first); continuing without them')
    FEATURES_LEAN = [f for f in FEATURES_LEAN if f in FEATURES_ALL]
print(f'feature sets: neural {len(FEATURES_LEAN)} | LightGBM {len(FEATURES_ALL)}')

## 0. Data, embeddings, history (as in Step 5)

Candidates are shuffled within each slate once, here, so no model can pick up list order (LightGBM's tie handling could otherwise; the neural model is order-invariant by construction).

In [ ]:
nU = len(pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user']))
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.02 if SMOKE else np.ones(nU, bool)
scope_users = np.flatnonzero(in_scope)
dset = ds.dataset(OUT_DIR / 'features.parquet')
flt = ds.field('user').isin(pa.array(scope_users.astype(np.int32))) if SMOKE else None
feat = dset.to_table(columns=['user', 'wid', 'label', 'split'] + FEATURES_ALL, filter=flt).to_pandas()
feat = feat.iloc[np.lexsort((np.random.default_rng(SEED).random(len(feat)), feat['user'].to_numpy()))]
feat = feat[feat.groupby('user')['label'].transform('max') > 0].reset_index(drop=True)
USER = feat['user'].to_numpy().astype(np.int64)
LABEL = feat['label'].to_numpy().astype(np.int64)
SPLIT = feat['split'].to_numpy()
print(f'{len(feat):,} candidate rows | slates: ' +
      ', '.join(f'{s} {feat.loc[SPLIT == s, "user"].nunique():,}' for s in ['train', 'val', 'test']))

In [ ]:
sfx = '_smoke' if SMOKE else ''
CACHE = {k: OUT_DIR / f'lsa{K}_{k}{sfx}.npy' for k in ('wids', 'item', 'query')}

def safe_save(path, arr):
    tmp = path.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        np.save(f, arr)
    np.load(tmp, mmap_mode='r')                       # verify it reads back before replacing
    os.replace(tmp, path)

def l2rows(M):
    n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1
    return M / n

def build_lsa():
    cw = pd.read_parquet(OUT_DIR / 'candidates.parquet', columns=['user', 'wid'])
    ew = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'wid', 'type'])
    ew = ew[ew['type'] < 3]
    need = np.union1d(cw['wid'].to_numpy()[in_scope[cw['user'].to_numpy()]],
                      ew['wid'].to_numpy()[in_scope[ew['user'].to_numpy()]])
    del cw, ew; gc.collect()

    wid_parts, doc_parts, tok_parts, off = [], [], [], 0
    reader = pacsv.open_csv(DATA_DIR / 'product_meta_data.txt',
        parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
        convert_options=pacsv.ConvertOptions(include_columns=['wid', 'name'],
                                             column_types={'wid': pa.int64(), 'name': pa.string()}))
    for batch in reader:
        tb = pa.Table.from_batches([batch])
        wid = tb['wid'].to_numpy()
        j = np.searchsorted(need, wid)
        keep = (j < len(need)) & (need[np.minimum(j, len(need) - 1)] == wid)
        if not keep.any():
            continue
        tb = tb.filter(pa.array(keep))
        parts = pc.split_pattern(pc.fill_null(tb['name'], '-1'), '\x18')
        flat = pc.list_flatten(parts)
        flat = pc.if_else(pc.equal(flat, ''), '-1', flat)
        ids = np.arange(off, off + len(tb), dtype=np.int64); off += len(tb)
        wid_parts.append(tb['wid'].to_numpy())
        doc_parts.append(np.repeat(ids, pc.list_value_length(parts).to_numpy()))
        tok_parts.append(pc.cast(flat, pa.int64()).to_numpy())
    wids = np.concatenate(wid_parts)
    d_, t_ = np.concatenate(doc_parts), np.concatenate(tok_parts)
    m = t_ != -1; d_, t_ = d_[m], t_[m]
    nD = len(wids)

    vocab = np.unique(t_)
    key = np.unique(d_ * len(vocab) + np.searchsorted(vocab, t_))
    d_, c_ = key // len(vocab), key % len(vocab)
    idf = (np.log((nD + 1) / (np.bincount(c_, minlength=len(vocab)) + 1)) + 1).astype(np.float32)
    norm = np.sqrt(np.bincount(d_, weights=idf[c_] ** 2, minlength=nD)); norm[norm == 0] = 1
    A = sp.csr_matrix((idf[c_] / norm[d_], (d_, c_)), shape=(nD, len(vocab)), dtype=np.float32)
    del d_, c_, key, t_; gc.collect()
    v0 = np.random.default_rng(SEED).random(min(A.shape)).astype(np.float32)
    _, S, Vt = svds(A, k=K, v0=v0, return_singular_vectors='vh')
    Vt = Vt[np.argsort(-S)].astype(np.float32)
    item = np.empty((nD, K), np.float32)
    for s in range(0, nD, 1_000_000):
        item[s:s + 1_000_000] = A[s:s + 1_000_000] @ Vt.T
    item = l2rows(item).astype(np.float16)
    del A; gc.collect()

    users = pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user', 'qry'])
    queries = pd.read_parquet(OUT_DIR / 'queries.parquet')
    qs = users['qry'].map(queries.set_index('qry')['text']).str.split('\x18')
    q_doc = np.repeat(np.arange(len(users), dtype=np.int64), qs.str.len().to_numpy())
    q_tok = pd.to_numeric(qs.explode(), errors='coerce').fillna(-1).to_numpy(np.int64)
    pos = np.searchsorted(vocab, q_tok)
    ok = (q_tok != -1) & (pos < len(vocab)) & (vocab[np.minimum(pos, len(vocab) - 1)] == q_tok)
    kq = np.unique(q_doc[ok] * len(vocab) + pos[ok])
    Q = sp.csr_matrix((idf[kq % len(vocab)], (kq // len(vocab), kq % len(vocab))),
                      shape=(len(users), len(vocab)), dtype=np.float32)
    qemb = l2rows(Q @ Vt.T).astype(np.float16)
    return wids, item, qemb

t0 = time.time()
if all(p.exists() for p in CACHE.values()):
    emb_wids, item_emb, q_emb = (np.load(CACHE[k]) for k in ('wids', 'item', 'query'))
    print(f'loaded cached embeddings in {time.time() - t0:.0f}s')
else:
    emb_wids, item_emb, q_emb = build_lsa()
    for k, a in zip(('wids', 'item', 'query'), (emb_wids, item_emb, q_emb)):
        safe_save(CACHE[k], a)
    print(f'built and cached embeddings in {time.time() - t0:.0f}s')
print(f'item embeddings {item_emb.shape} | query embeddings {q_emb.shape}')
emb_index = pd.Index(emb_wids)
DOC = emb_index.get_indexer(feat['wid']) + 1            # 0 = no name / unknown (zero vector)

In [ ]:
ev = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'pos', 'wid', 'type', 'age_days'])
ev = ev[(ev['type'] < 3) & in_scope[ev['user'].to_numpy()]]
ev = ev.sort_values(['user', 'pos'], kind='stable')
slot = ev.groupby('user').cumcount(ascending=False).to_numpy()      # 0 = most recent
ev, slot = ev[slot < L_MAX], slot[slot < L_MAX]
u_ev = ev['user'].to_numpy()
H_DOC = np.zeros((nU, L_MAX), np.int64); H_TYPE = np.zeros((nU, L_MAX), np.int64)
H_AGE = np.zeros((nU, L_MAX), np.float32); H_MASK = np.zeros((nU, L_MAX), bool)
H_DOC[u_ev, slot] = emb_index.get_indexer(ev['wid']) + 1
H_TYPE[u_ev, slot] = ev['type'].to_numpy()
H_AGE[u_ev, slot] = np.log1p(ev['age_days'].to_numpy())
H_MASK[u_ev, slot] = True
del ev, slot, u_ev; gc.collect()

used = np.unique(np.r_[0, DOC, H_DOC[H_MASK]])
E_full = np.vstack([np.zeros((1, K), np.float16), item_emb])
E = torch.from_numpy(E_full[used]).to(DEVICE)
DOC = np.searchsorted(used, DOC); H_DOC = np.searchsorted(used, H_DOC)
QE = torch.from_numpy(q_emb).to(DEVICE)
del E_full, item_emb; gc.collect()

def build_X(cols):
    X = feat[cols].to_numpy(np.float32, copy=True)
    nan = np.isnan(X)
    np.copysign(np.log1p(np.abs(X)), X, out=X)
    tr = SPLIT == 'train'
    mu = np.nanmean(X[tr], 0); sd = np.nanstd(X[tr], 0); sd[~(sd > 1e-6)] = 1; mu = np.nan_to_num(mu)
    X -= mu; X /= sd; X[nan] = 0
    return torch.from_numpy(np.concatenate([X, nan[:, nan.any(0)].astype(np.float32)], 1))

def make_split(name):
    idx = np.flatnonzero(SPLIT == name)
    u = USER[idx]
    start = np.r_[0, np.flatnonzero(u[1:] != u[:-1]) + 1]
    return {'idx': idx, 'start': start, 'size': np.diff(np.r_[start, len(idx)]), 'user': u[start]}

SPLITS = {s: make_split(s) for s in ['train', 'val', 'test']}
X_NN = build_X(FEATURES_LEAN)
LABEL_T, DOC_T = torch.from_numpy(LABEL), torch.from_numpy(DOC)
H_DOC_T, H_TYPE_T = torch.from_numpy(H_DOC), torch.from_numpy(H_TYPE)
H_AGE_T, H_MASK_T = torch.from_numpy(H_AGE), torch.from_numpy(H_MASK)
print(f'history up to {L_MAX} events | embedding table {E.shape[0]:,} x {K} on {DEVICE} | neural input width {X_NN.shape[1]}')

## Shared: per-slate metrics

`slate_metrics` returns one value per slate (sorted by user), so every model is scored on identical slates, and the paired bootstrap can compare them slate by slate. Metrics are identical to all earlier steps; *purchase rank* (position of the first purchased item, 1 = top) is added for the business translation.

In [ ]:
def slate_metrics(u, lab, score, k=10, seed=0):
    lab = lab.astype(np.float64)
    r = np.random.default_rng(seed).random(len(u))
    order = np.lexsort((r, -np.asarray(score, dtype=np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    size = np.diff(np.r_[start, len(u_s)])
    rank = np.arange(len(u_s)) - np.repeat(start, size)
    top = rank < k
    disc = np.where(top, 1.0 / np.log2(rank + 2), 0.0)
    gid = np.repeat(np.arange(len(start)), size)
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    idcg = np.bincount(gid, weights=(2 ** lab[np.lexsort((-lab, u))] - 1) * disc)
    has_buy = np.bincount(gid, weights=(lab_s == 3)) > 0
    buy_rank = np.minimum.reduceat(np.where(lab_s == 3, rank + 1, 10 ** 6), start).astype(float)
    buy_rank[~has_buy] = np.nan
    return {'ndcg': dcg / idcg, 'hit': np.bincount(gid, weights=((lab_s > 0) & top)) > 0,
            'has_buy': has_buy, 'buy_hit': np.bincount(gid, weights=((lab_s == 3) & top)) > 0, 'buy_rank': buy_rank}

def evaluate(u, lab, score):
    m = slate_metrics(u, lab, score)
    return {'NDCG@10': m['ndcg'].mean(), 'HR@10': m['hit'].mean(), 'Purchase-HR@10': m['buy_hit'][m['has_buy']].mean(),
            'median purchase rank': np.nanmedian(m['buy_rank']), 'slates': len(m['ndcg'])}

def split_eval(name, score):
    S = SPLITS[name]
    return evaluate(USER[S['idx']], LABEL[S['idx']], score)
print('metrics defined')

## 1. Neural ranker: longer training and a small grid

Architecture as Step 5 model C (32 features, embeddings, history attention, slate context, ordinal head), now with three tunable settings: history length *L*, width *d*, and the weight of the listwise loss term. Training runs up to 12 epochs with a cosine learning-rate decay and stops when validation NDCG@10 hasn't improved for 2 epochs.

In [ ]:
class Ranker(nn.Module):
    def __init__(self, n_feat, d=128, k=K, h=64):
        super().__init__()
        self.feat = nn.Sequential(nn.Linear(n_feat, d), nn.ReLU(), nn.Linear(d, d), nn.ReLU())
        self.item, self.qry = nn.Linear(k, h), nn.Linear(k, h)
        self.type_emb, self.age = nn.Embedding(3, 8), nn.Linear(1, 8)
        self.hist = nn.Linear(h + 16, h)
        self.att_h, self.att_c, self.att_p = nn.Linear(h, h), nn.Linear(h, h, bias=False), nn.Linear(h, h, bias=False)
        self.att_out = nn.Linear(h, 1)
        self.enc = nn.Sequential(nn.Linear(d + 3 * h, d), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(3 * d, d), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d, 1))
        self.theta0, self.dtheta = nn.Parameter(torch.tensor(4.0)), nn.Parameter(torch.tensor([0.87, 1.28]))

    def thresholds(self):
        return torch.cat([self.theta0.view(1), self.theta0 + torch.cumsum(Fn.softplus(self.dtheta), 0)])

    def forward(self, b):
        c = self.item(E[b['doc']].float())
        q = self.qry(QE[b['quser']].float())[:, None].expand(-1, c.shape[1], -1)
        hv = self.hist(torch.cat([self.item(E[b['hdoc']].float()), self.type_emb(b['htype']),
                                  self.age(b['hage'][..., None])], -1))
        pre = self.att_h(hv)[:, None] + self.att_c(c)[:, :, None] + self.att_p(hv[:, None] * c[:, :, None])
        a = self.att_out(torch.relu(pre)).squeeze(-1).float().masked_fill(~b['hmask'][:, None, :], -1e4)
        w = torch.softmax(a, -1) * b['hmask'].any(-1)[:, None, None]
        interest = torch.einsum('bml,bld->bmd', w.to(hv.dtype), hv)
        z = self.enc(torch.cat([self.feat(b['X']), c, q, interest], -1))
        m = b['mask'][..., None].to(z.dtype)
        ctx = ((z * m).sum(1, keepdim=True) / m.sum(1, keepdim=True).clamp(min=1)).expand_as(z)
        return self.head(torch.cat([z, ctx, z - ctx], -1)).squeeze(-1)

def make_batch(S, slates, L):
    st, sz = S['start'][slates], S['size'][slates]
    B, M = len(slates), int(sz.max())
    local = np.concatenate([np.arange(a, a + n) for a, n in zip(st, sz)])
    rows = S['idx'][local]; rows_t = torch.from_numpy(rows)
    bi = torch.from_numpy(np.repeat(np.arange(B), sz)); pi = torch.from_numpy(local - np.repeat(st, sz))
    def pad(src):
        out = torch.zeros((B, M) + tuple(src.shape[1:]), dtype=src.dtype)
        out[bi, pi] = src[rows_t]
        return out
    users = S['user'][slates]
    mask = torch.zeros(B, M, dtype=torch.bool); mask[bi, pi] = True
    b = {'X': pad(X_NN), 'doc': pad(DOC_T), 'y': pad(LABEL_T), 'mask': mask, 'quser': torch.from_numpy(users),
         'hdoc': H_DOC_T[users, :L], 'htype': H_TYPE_T[users, :L], 'hage': H_AGE_T[users, :L], 'hmask': H_MASK_T[users, :L]}
    b = {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}
    b['rows'] = rows
    return b

def loss_fn(s, b, theta, lw):
    m, y = b['mask'], b['y']
    tgt = (y[..., None] >= torch.arange(1, 4, device=s.device)).float()
    bce = Fn.binary_cross_entropy_with_logits(s[..., None] - theta, tgt, reduction='none').sum(-1)
    bce = (bce * m).sum() / m.sum()
    gain = (2.0 ** y - 1) * m
    p = gain / gain.sum(1, keepdim=True).clamp(min=1e-9)
    return bce + lw * -(p * torch.log_softmax(s.masked_fill(~m, -1e4), 1)).sum(1).mean()

@torch.no_grad()
def ordinal_probs(model, name, L, bs=128):
    """P(Y>=1), P(Y>=2), P(Y>=3) for every row of a split, in split order."""
    model.eval(); S = SPLITS[name]
    s_out = np.empty(len(S['idx']), np.float32)
    pos_of = np.empty(len(USER), np.int64); pos_of[S['idx']] = np.arange(len(S['idx']))
    for i in range(0, len(S['start']), bs):
        b = make_batch(S, np.arange(i, min(i + bs, len(S['start']))), L)
        with torch.autocast(device_type=AMP_DEV, dtype=torch.float16, enabled=USE_AMP):
            s = model(b)
        s_out[pos_of[b['rows']]] = s.float()[b['mask']].cpu().numpy()
    theta = model.thresholds().detach().float().cpu().numpy()
    return 1 / (1 + np.exp(-(s_out[:, None] - theta)))

def train_nn(cfg, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = Ranker(X_NN.shape[1], d=cfg['d']).to(DEVICE)
    TR = SPLITS['train']; steps = int(np.ceil(len(TR['start']) / BATCH))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS * steps)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    best, best_state, bad, log = -1.0, None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train(); t0 = time.time()
        perm = rng.permutation(len(TR['start']))
        for i in range(0, len(perm), BATCH):
            b = make_batch(TR, perm[i:i + BATCH], cfg['L'])
            with torch.autocast(device_type=AMP_DEV, dtype=torch.float16, enabled=USE_AMP):
                s = model(b)
            loss = loss_fn(s.float(), b, model.thresholds(), cfg['lw'])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        val = split_eval('val', ordinal_probs(model, 'val', cfg['L']).sum(1))['NDCG@10']
        log.append(val)
        print(f'    epoch {ep + 1}: val NDCG@10 {val:.4f} ({(time.time() - t0) / 60:.1f} min)')
        if val > best + 1e-4:
            best, best_state, bad = val, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best, len(log)

NN_BASE = {'L': 50, 'd': 128, 'lw': 1.0, 'lr': 1e-3}
GRID = {   # comment out entries to save GPU time
    'base (L=50, d=128, listwise 1.0)': {},
    'longer history (L=100)':           {'L': 100},
    'wider (d=256)':                    {'d': 256},
    'listwise weight 2.0':              {'lw': 2.0},
}
if SMOKE:
    GRID = dict(list(GRID.items())[:2])
nn_grid, nn_probs = {}, {}
for name, delta in GRID.items():
    cfg = {**NN_BASE, **delta}
    print(f'== {name}')
    t0 = time.time()
    model, best, n_ep = train_nn(cfg, SEEDS[0])
    nn_grid[name] = {'val NDCG@10': best, 'epochs': n_ep, 'min': (time.time() - t0) / 60, **cfg}
    nn_probs[(name, SEEDS[0])] = {s: ordinal_probs(model, s, cfg['L']) for s in ('val', 'test')}
    del model; gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()
pd.DataFrame(nn_grid).T.round(4)

In [ ]:
best_nn = max(nn_grid, key=lambda n: nn_grid[n]['val NDCG@10'])
best_cfg = {k: nn_grid[best_nn][k] for k in NN_BASE}
print(f'best setting on validation: {best_nn} {best_cfg}')
for seed in SEEDS[1:]:
    print(f'== {best_nn}, seed {seed}')
    model, _, _ = train_nn(best_cfg, seed)
    nn_probs[(best_nn, seed)] = {s: ordinal_probs(model, s, best_cfg['L']) for s in ('val', 'test')}
    del model; gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()

seed_rows = {seed: split_eval('test', nn_probs[(best_nn, seed)]['test'].sum(1)) for seed in SEEDS}
seed_df = pd.DataFrame(seed_rows).T
print('single-seed test results (score = expected interaction level E[Y] = sum of the three probabilities):')
print(seed_df[['NDCG@10', 'HR@10', 'Purchase-HR@10']].round(4).to_string())
print('mean ± sd: ' + ', '.join(f'{c} {seed_df[c].mean():.4f} ± {seed_df[c].std(ddof=1):.4f}'
                                for c in ['NDCG@10', 'HR@10', 'Purchase-HR@10']))

# the final neural model averages the seeds' probabilities
NN_P = {s: np.mean([nn_probs[(best_nn, seed)][s] for seed in SEEDS], 0) for s in ('val', 'test')}
NN_SCORE = {s: NN_P[s].sum(1) for s in NN_P}
print(f"seed-averaged neural model: val NDCG@10 {split_eval('val', NN_SCORE['val'])['NDCG@10']:.4f}")

## 2. LightGBM: purchase-weighted objective and light tuning

LambdaRank's `label_gain` sets how much credit each label level earns in the NDCG it optimises; the default `[0, 1, 3, 7]` matches 2^label − 1. Raising the purchase gain to 15 or 31 tells the model that ranking purchases first matters more, which targets Purchase-HR@10, the metric closest to session conversion. The best gain is then combined with two hyperparameter variants. All variants use the 55 features, as in Step 4b; the default variant re-trains the benchmark.

In [ ]:
def lgb_data(name):
    S = SPLITS[name]
    return feat.iloc[S['idx']][FEATURES_ALL], LABEL[S['idx']], S['size']

X_tr, y_tr, g_tr = lgb_data('train'); X_va, y_va, g_va = lgb_data('val'); X_te, _, _ = lgb_data('test')
LGB_BASE = dict(objective='lambdarank', n_estimators=LGB_TREES, learning_rate=0.05, num_leaves=63,
                min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                random_state=SEED, n_jobs=-1, verbose=-1, label_gain=[0, 1, 3, 7])

def fit_lgb(params):
    m = lgb.LGBMRanker(**params)
    m.fit(X_tr, y_tr, group=g_tr, eval_set=[(X_va, y_va)], eval_group=[g_va], eval_at=[10],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return m

lgb_grid, lgb_scores = {}, {}
def run_lgb(name, delta):
    params = {**LGB_BASE, **delta}
    t0 = time.time(); m = fit_lgb(params)
    sv, st = m.predict(X_va), m.predict(X_te)
    lgb_scores[name] = {'val': sv, 'test': st}
    lgb_grid[name] = {**{f'val {k}': v for k, v in split_eval('val', sv).items() if k in ('NDCG@10', 'Purchase-HR@10')},
                      'trees': m.best_iteration_, 'min': (time.time() - t0) / 60}
    print(f"  {name}: val NDCG@10 {lgb_grid[name]['val NDCG@10']:.4f} | val Purchase-HR@10 "
          f"{lgb_grid[name]['val Purchase-HR@10']:.4f} | {lgb_grid[name]['min']:.1f} min")

for g in [7, 15, 31]:
    run_lgb(f'label_gain purchase={g}', {'label_gain': [0, 1, 3, g]})
best_gain = max((n for n in lgb_grid if n.startswith('label_gain')), key=lambda n: lgb_grid[n]['val NDCG@10'])
gain_delta = {'label_gain': [0, 1, 3, int(best_gain.split('=')[1])]}
run_lgb(f'{best_gain}, num_leaves 127, min_child 100', {**gain_delta, 'num_leaves': 127, 'min_child_samples': 100})
run_lgb(f'{best_gain}, lr 0.03, more trees', {**gain_delta, 'learning_rate': 0.03, 'n_estimators': 2 * LGB_TREES})

LGB_BENCH = 'label_gain purchase=7'
best_lgb = max(lgb_grid, key=lambda n: lgb_grid[n]['val NDCG@10'])
print(f'\nbest LightGBM on validation: {best_lgb}')
del X_tr, y_tr; gc.collect()
pd.DataFrame(lgb_grid).T.round(4)

## 3. Ensemble

The two models' scores live on different scales, so within each slate each score is converted either to a z-score or to a percentile rank, then combined as w · neural + (1 − w) · LightGBM. The method and w are chosen on validation only.

In [ ]:
def within_slate(name, x, how):
    S = SPLITS[name]; st, sz = S['start'], S['size']
    if how == 'rank':
        u = np.repeat(np.arange(len(st)), sz)
        return pd.Series(x).groupby(u).rank(pct=True).to_numpy()
    mu = np.add.reduceat(x, st) / sz
    sd = np.sqrt(np.maximum(np.add.reduceat(x * x, st) / sz - mu ** 2, 0)) + 1e-9
    return (x - np.repeat(mu, sz)) / np.repeat(sd, sz)

LGB_SCORE = {s: lgb_scores[best_lgb][s] for s in ('val', 'test')}
W = np.round(np.linspace(0, 1, 11), 1)
ens_val = {}
for how in ['z-score', 'rank']:
    a, b = within_slate('val', NN_SCORE['val'], how), within_slate('val', LGB_SCORE['val'], how)
    for w in W:
        ens_val[(how, w)] = split_eval('val', w * a + (1 - w) * b)['NDCG@10']
best_how, best_w = max(ens_val, key=ens_val.get)
ENS_SCORE = {s: best_w * within_slate(s, NN_SCORE[s], best_how) + (1 - best_w) * within_slate(s, LGB_SCORE[s], best_how)
             for s in ('val', 'test')}
curve = pd.DataFrame({how: [ens_val[(how, w)] for w in W] for how in ['z-score', 'rank']}, index=W)
curve.index.name = 'weight on neural model'
print(f'chosen on validation: {best_how}, w = {best_w} (val NDCG@10 {ens_val[(best_how, best_w)]:.4f})')
curve.round(4)

## 4. Final test results and significance

First use of the test set. Paired bootstrap: slates are resampled 2,000 times with replacement, and each comparison's mean difference in per-slate NDCG@10 (or purchase hit) is recomputed on the same resampled slates for both models. The 95% interval and one-sided p-value come from that distribution.

In [ ]:
TE = SPLITS['test']; te_u, te_lab = USER[TE['idx']], LABEL[TE['idx']]
pop = feat['i_users_ord_365d'].to_numpy()[TE['idx']]
FINAL = {
    'random': np.zeros(len(te_u)),
    'popularity (orders, 365 d)': np.nan_to_num(pop, nan=-1.0),
    'LightGBM benchmark (retrained)': lgb_scores[LGB_BENCH]['test'],
    f'LightGBM tuned ({best_lgb})': LGB_SCORE['test'],
    f'neural, {len(SEEDS)}-seed average ({best_nn})': NN_SCORE['test'],
    f'ensemble ({best_how}, w={best_w})': ENS_SCORE['test'],
}
M = {n: slate_metrics(te_u, te_lab, s) for n, s in FINAL.items()}
final = pd.DataFrame({n: evaluate(te_u, te_lab, s) for n, s in FINAL.items()}).T
print(final[['NDCG@10', 'HR@10', 'Purchase-HR@10', 'median purchase rank']].round(4).to_string())

def paired(a, b, metric, n_boot=2000, seed=SEED):
    ma, mb = M[a], M[b]
    if metric == 'ndcg':
        d = ma['ndcg'] - mb['ndcg']
    else:
        keep = ma['has_buy']
        d = ma['buy_hit'][keep].astype(float) - mb['buy_hit'][keep].astype(float)
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0, len(d), (200, len(d)))].mean(1) for _ in range(n_boot // 200)])
    return {'difference': d.mean(), 'ci_low': np.percentile(boot, 2.5), 'ci_high': np.percentile(boot, 97.5),
            'p (one-sided)': max((boot <= 0).mean(), 1 / n_boot)}

names = list(FINAL)
bench, lgb_t, nn_n, ens = names[2], names[3], names[4], names[5]
tests = {}
for a, b in [(ens, bench), (ens, lgb_t), (ens, nn_n), (nn_n, bench), (lgb_t, bench)]:
    for metric, label in [('ndcg', 'NDCG@10'), ('buy', 'Purchase-HR@10')]:
        tests[f'{a.split(" (")[0]} vs {b.split(" (")[0]} [{label}]'] = paired(a, b, metric)
print('\npaired bootstrap over test slates (positive difference = first model better):')
pd.DataFrame(tests).T.round(5)

In [ ]:
hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
te_wid = feat['wid'].to_numpy()[TE['idx']]
j = np.searchsorted(hist_wids, te_wid)
cold = (j >= len(hist_wids)) | (hist_wids[np.minimum(j, len(hist_wids) - 1)] != te_wid)
seg_user = pd.cut(pd.Series(np.where(te_lab > 0, cold, np.nan)).groupby(te_u).mean(),
                  [-0.01, 0.0, 0.999, 1.0], labels=['all warm', 'mixed', 'all cold'])
seg = pd.Series(te_u).map(seg_user).to_numpy()
seg_table = pd.DataFrame({n: {s: evaluate(te_u[seg == s], te_lab[seg == s], FINAL[n][seg == s])['NDCG@10']
                              for s in ['all warm', 'mixed', 'all cold']} for n in names[2:]}).T
print('test NDCG@10 by segment (slates whose positives are all warm / mixed / all cold items):')
seg_table.round(4)

## 5. Business translation

**Session conversion estimate.** The neural model's calibrated P(purchase) per candidate gives each session a conversion probability, 1 − Π(1 − P(purchase)ᵢ), treating candidates as independent. If these estimates match observed conversion by decile, they can be used to target promotions at sessions unlikely to convert (Step 4c showed promotion-dependent product types convert 2.5× worse).

**Where the purchased item lands.** For sessions that ended in a purchase: how often each ranking puts the purchased item in the top 10, and its median position. JD's original list order can't serve as a baseline because it leaks the labels (interim report §2.2.1).

In [ ]:
P3 = NN_P['test'][:, 2]
st, sz = TE['start'], TE['size']
p_conv = 1 - np.exp(np.add.reduceat(np.log1p(-np.clip(P3, 0, 1 - 1e-7)), st))
obs_conv = np.maximum.reduceat((te_lab == 3).astype(int), st)
dec = pd.qcut(pd.Series(p_conv).rank(method='first'), 10, labels=[f'D{i}' for i in range(1, 11)])
calib = pd.DataFrame({'predicted': p_conv, 'observed': obs_conv, 'decile': dec}).groupby('decile', observed=True).agg(
    sessions=('observed', 'size'), predicted_conversion=('predicted', 'mean'), observed_conversion=('observed', 'mean'))
print(f'test sessions with an interaction: {len(st):,} | predicted conversion {p_conv.mean():.2%} vs observed {obs_conv.mean():.2%}')
print('by decile of predicted conversion (D10 = most likely to convert):')
print((calib * [1, 100, 100]).round(2).rename(columns=lambda c: c + (' %' if c != 'sessions' else '')).to_string())

biz = pd.DataFrame({n: {'purchased item in top 10': M[n]['buy_hit'][M[n]['has_buy']].mean(),
                        'median position of purchased item': np.nanmedian(M[n]['buy_rank']),
                        'purchased item at position 1': (M[n]['buy_rank'] == 1)[M[n]['has_buy']].mean()}
                    for n in names}).T
print(f"\nsessions that ended in a purchase: {int(M[names[0]]['has_buy'].sum()):,}")
biz.round(4)

In [ ]:
res = pd.concat([final.assign(stage='final test'),
                 pd.DataFrame(nn_grid).T.assign(stage='neural grid (val)'),
                 pd.DataFrame(lgb_grid).T.assign(stage='LightGBM grid (val)'),
                 pd.DataFrame(tests).T.assign(stage='paired bootstrap (test)')])
res.index.name = 'model / comparison'
pred = pd.DataFrame({'user': te_u, 'wid': te_wid, 'label': te_lab, 'lgbm': LGB_SCORE['test'], 'neural': NN_SCORE['test'],
                     'ensemble': ENS_SCORE['test'], 'p_click_plus': NN_P['test'][:, 0],
                     'p_cart_plus': NN_P['test'][:, 1], 'p_purchase': NN_P['test'][:, 2]})
sfx = '_smoke' if SMOKE else ''
for path, write in [(OUT_DIR / f'step6_results{sfx}.csv', lambda p: res.to_csv(p)),
                    (OUT_DIR / f'step6_test_predictions{sfx}.parquet', lambda p: pred.to_parquet(p, index=False))]:
    tmp = path.with_name('tmp_' + path.name)
    write(tmp); os.replace(tmp, path)
    print('saved', path)
try:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted - safe to close the runtime')
except NameError:
    pass

## 6. Takeaways

*(filled in after the run)*